# 検証: 「上位ランキングの日本酒は甘口で芳醇なものが多い」は本当か？

SAKETIMEランキングの上位銘柄は甘口・芳醇に偏っているという主張を、
**構造化データ（テイスト評価）**と**レビューテキスト分析**の両面から検証する。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import numpy as np
import warnings
warnings.filterwarnings('ignore', message='Glyph.*missing from font')

plt.style.use('seaborn-v0_8-whitegrid')
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = ['Hiragino Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

ranking = pd.read_csv('../data/ranking.csv')
reviews = pd.read_csv('../data/reviews.csv')
rev = reviews.merge(ranking[['brand_id','rank','name','rating']], on='brand_id', how='left', suffixes=('_review','_brand'))
rev = rev.copy()
rev['tier'] = pd.cut(rev['rank'], bins=[0,10,20,50,100,500,5386],
                     labels=['Top10','11-20','21-50','51-100','101-500','501+'])
print(f'レビュー件数: {len(rev):,}')

## 1. 構造化データによる検証（ユーザーが選択した甘辛・ボディ）

In [ ]:
# ランキング帯別の甘口率・濃醇率を計算（501+を追加）
tiers = ['Top10','11-20','21-50','51-100','101-500','501+']
sweet_rates = []
dry_rates = []
rich_rates = []
light_rates = []

for tier in tiers:
    t = rev[rev['tier'] == tier]
    s = t.dropna(subset=['sweetness'])
    b = t.dropna(subset=['body'])
    sweet_rates.append(s['sweetness'].isin(['甘い+1','甘い+2']).mean() * 100)
    dry_rates.append(s['sweetness'].isin(['辛い+1','辛い+2']).mean() * 100)
    rich_rates.append(b['body'].isin(['重い+1','重い+2']).mean() * 100)
    light_rates.append(b['body'].isin(['軽い+1','軽い+2']).mean() * 100)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

x = np.arange(len(tiers))
w = 0.35
axes[0].bar(x - w/2, sweet_rates, w, label='甘口（甘い+1, +2）', color='#E91E63')
axes[0].bar(x + w/2, dry_rates, w, label='辛口（辛い+1, +2）', color='#2196F3')
axes[0].set_xticks(x)
axes[0].set_xticklabels(tiers)
axes[0].set_ylabel('割合 (%)')
axes[0].set_title('ランキング帯別: 甘口 vs 辛口')
axes[0].legend()
axes[0].set_ylim(0, 75)
for i, v in enumerate(sweet_rates):
    axes[0].text(i - w/2, v + 1, f'{v:.0f}%', ha='center', fontsize=8)
for i, v in enumerate(dry_rates):
    axes[0].text(i + w/2, v + 1, f'{v:.0f}%', ha='center', fontsize=8)

axes[1].bar(x - w/2, rich_rates, w, label='濃醇（重い+1, +2）', color='#FF9800')
axes[1].bar(x + w/2, light_rates, w, label='軽快（軽い+1, +2）', color='#4CAF50')
axes[1].set_xticks(x)
axes[1].set_xticklabels(tiers)
axes[1].set_ylabel('割合 (%)')
axes[1].set_title('ランキング帯別: 濃醇 vs 軽快')
axes[1].legend()
axes[1].set_ylim(0, 45)
for i, v in enumerate(rich_rates):
    axes[1].text(i - w/2, v + 1, f'{v:.0f}%', ha='center', fontsize=8)
for i, v in enumerate(light_rates):
    axes[1].text(i + w/2, v + 1, f'{v:.0f}%', ha='center', fontsize=8)

plt.suptitle('構造化データによる検証: 上位銘柄ほど甘口だが、芳醇（濃醇）ではない', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../data/verification_structured.png', dpi=150, bbox_inches='tight')
plt.show()

### 構造化データの結論

- **甘口は正しい**: Top10の甘口率67%に対し、101-500位は35%。ランキング上位ほど明確に甘口
- **芳醇（濃醇）は逆**: Top10の濃醇率11%に対し、101-500位は20%。**上位ほどむしろ軽快**（軽い+1,+2が34%）

## 2. レビューテキストによる検証

In [ ]:
# ランキング帯別のキーワード出現率（501+を追加）
categories = {
    '甘口系\n(甘い/甘み/甘味/甘さ)': '甘い|甘み|甘味|甘口|甘さ|甘やか',
    '辛口系\n(辛い/辛口/ドライ/キレ)': '辛い|辛口|辛味|ドライ',
    '芳醇系\n(芳醇/旨味/濃厚/コク/リッチ)': '芳醇|濃醇|コク|旨味|旨み|濃厚|重厚|リッチ',
    '軽快系\n(すっきり/爽やか/クリア/軽い)': '淡麗|軽い|軽快|すっきり|スッキリ|爽やか|さっぱり|クリア|シャープ',
    'フルーティ系\n(果実/メロン/りんご等)': 'フルーティ|フルーツ|果実|メロン|りんご|リンゴ|パイナップル|マスカット|ライチ|洋梨|バナナ|マンゴー|桃',
    '上品・エレガント系\n(上品/綺麗/繊細)': '上品|エレガント|綺麗|品のある|優雅|繊細',
}

rev_with_comment = rev.dropna(subset=['comment'])
results = {}
for cat_name, pattern in categories.items():
    rates = []
    for tier in tiers:
        tier_data = rev_with_comment[rev_with_comment['tier'] == tier]
        rate = tier_data['comment'].str.contains(pattern, na=False).mean() * 100
        rates.append(rate)
    results[cat_name] = rates

fig, ax = plt.subplots(figsize=(14, 7))
x = np.arange(len(tiers))
width = 0.13
colors = ['#E91E63', '#2196F3', '#FF9800', '#4CAF50', '#9C27B0', '#795548']

for i, (cat, rates) in enumerate(results.items()):
    ax.bar(x + (i - 2.5) * width, rates, width, label=cat, color=colors[i], alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(tiers, fontsize=11)
ax.set_ylabel('レビュー中の出現率 (%)', fontsize=11)
ax.set_title('レビューテキスト中の味わいキーワード出現率（ランキング帯別）', fontsize=13)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
ax.set_ylim(0, 40)

plt.tight_layout()
plt.savefig('../data/verification_text.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Top10銘柄の個別テイストレーダーチャート
from matplotlib.patches import FancyBboxPatch

keyword_groups = {
    '甘口': '甘い|甘み|甘味|甘口|甘さ|甘やか',
    '辛口': '辛い|辛口|ドライ',
    '芳醇': '芳醇|旨味|旨み|濃厚|コク|リッチ',
    '軽快': 'すっきり|スッキリ|爽やか|軽い|クリア',
    'フルーティ': 'フルーティ|フルーツ|果実|メロン|りんご|パイナップル|マスカット|バナナ|マンゴー',
    '上品': '上品|エレガント|綺麗|繊細',
}

fig, axes = plt.subplots(2, 5, figsize=(20, 8), subplot_kw=dict(polar=True))
categories_list = list(keyword_groups.keys())
N = len(categories_list)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

for idx, (_, row) in enumerate(ranking.head(10).iterrows()):
    ax = axes[idx // 5][idx % 5]
    brand_rev = rev_with_comment[rev_with_comment['brand_id'] == row['brand_id']]
    
    values = []
    for kw_name, pattern in keyword_groups.items():
        val = brand_rev['comment'].str.contains(pattern, na=False).mean() * 100
        values.append(val)
    values += values[:1]
    
    ax.plot(angles, values, 'o-', linewidth=2, color='#E91E63')
    ax.fill(angles, values, alpha=0.15, color='#E91E63')
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories_list, fontsize=7)
    ax.set_title(f'{idx+1}位 {row["name"]}\n({row["rating"]})', fontsize=10, pad=15)
    ax.set_ylim(0, 50)

plt.suptitle('Top10銘柄の味わいプロファイル（レビューテキスト分析）', fontsize=14, y=1.03)
plt.tight_layout()
plt.savefig('../data/verification_radar.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. 甘辛 × 芳醇/軽快のクロス分析

In [ ]:
# 構造化データ: 甘辛 × ボディのヒートマップ（ランキング帯別、501+を追加）
fig, axes = plt.subplots(1, 4, figsize=(22, 5))

body_order = ['軽い+2', '軽い+1', '普通', '重い+1', '重い+2']
sweet_order = ['辛い+2', '辛い+1', '普通', '甘い+1', '甘い+2']

for i, (tier_name, tier_range) in enumerate([('Top 10', (1,10)), ('51-100位', (51,100)), ('101-500位', (101,500)), ('501位以降', (501,5386))]):
    tier = rev[(rev['rank'] >= tier_range[0]) & (rev['rank'] <= tier_range[1])]
    taste = tier.dropna(subset=['body', 'sweetness'])
    cross = pd.crosstab(taste['body'], taste['sweetness'], normalize='all') * 100
    cross = cross.reindex(index=[b for b in body_order if b in cross.index],
                          columns=[s for s in sweet_order if s in cross.columns])
    
    sns.heatmap(cross, annot=True, fmt='.1f', cmap='YlOrRd', ax=axes[i],
                vmin=0, vmax=35, cbar_kws={'label': '%'})
    axes[i].set_title(f'{tier_name} (n={len(taste):,})', fontsize=12)
    axes[i].set_xlabel('甘辛')
    axes[i].set_ylabel('ボディ' if i == 0 else '')

plt.suptitle('テイスト分布の比較: ランキング帯別（各セル = 全体に占める%）', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../data/verification_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. 総合結論

In [ ]:
# 結論をデータで裏付け
print('=' * 70)
print('検証結果: 「上位ランキングの日本酒は甘口で芳醇」は部分的に正しい')
print('=' * 70)

print('''
■ 甘口 → ✅ 正しい
  - 構造化データ: Top10の甘口率67.3% vs 101-500位の35.4%（約2倍）
  - テキスト分析: 甘口系キーワード出現率 Top10で32.8% vs 101-500位で28.5%
  - 逆に辛口系は Top10で13.8% vs 101-500位で21.1%（上位ほど少ない）
  - Top10の十四代、陽乃鳥、宮寒梅、花陽浴は特に甘口傾向が顕著

■ 芳醇 → ❌ むしろ逆
  - 構造化データ: Top10の濃醇率11.2% vs 101-500位の19.5%（上位ほど軽い）
  - テキスト分析: 芳醇系キーワード Top10で18.4% vs 101-500位で25.0%
  - 軽快系キーワードもTop10で16.7% vs 101-500位で22.9%（差は小さい）
  - 上位銘柄の特徴は「濃醇」ではなく「普通〜やや軽め」のボディ

■ より正確な表現
  上位ランキングの日本酒は「甘口でフルーティ」なものが多い。
  ボディは芳醇というより「普通〜軽めで上品」な傾向。
  Top10ではフルーティ系キーワードが25.7%と高く、
  花陽浴(36.7%)、電照菊(26.6%)、宮寒梅(25.9%)が特に顕著。
''')